# Notebook 4 — NumPy: Fast Maths on Arrays

**Curriculum position:** Containers → Control flow → Packaging logic → **NumPy** → Matplotlib → SciPy

NumPy (Numerical Python) is the foundation of almost every scientific computing
workflow in Python. Once you understand it, everything else — plotting, statistics,
simulation — feels natural. This notebook builds that foundation from scratch.

**What you will cover:**
1. Creating Arrays — building the data structure that makes NumPy fast
2. Indexing and Slicing Arrays — reaching inside arrays to pull out what you need
3. Array Operations and Broadcasting — doing maths on entire arrays at once
4. Aggregation and Statistical Functions — summarising data with one call
5. Random Numbers with numpy.random — generating simulated data

**Before you start:** run the cell below to import NumPy. The alias `np` is
universal — every NumPy book and tutorial uses it, so adopt it immediately.


In [ ]:
import numpy as np   # np is the standard alias; you'll see it everywhere
print("NumPy version:", np.__version__)


---
# Item 1 — Creating Arrays

> **By the end of this you'll be able to:** explain why NumPy arrays exist,
> create arrays from lists, from scratch (zeros, ones, full), and build evenly
> spaced sequences with `arange` and `linspace`. Everything below the core
> sections is optional on a first pass.

## Why NumPy exists — and why plain Python lists aren't enough

You already know Python lists. They're flexible: one list can hold an integer,
a string, and a float all at once. That flexibility has a cost. When Python
stores a list it has to remember the type of every single element separately,
and when you do maths on a list you have to loop through element by element.
For ten items that's fine. For a recording of ten thousand voltage readings
taken every millisecond, it becomes painfully slow.

**Illustration 1 — a ruled measuring tape vs a pile of sticky notes.**
A measuring tape is a single rigid strip; every centimetre mark is the same
kind of thing in the same format, so you can read the whole strip in one glance.
A pile of sticky notes can hold anything — phone numbers, doodles, reminders —
but you can't do maths on a pile of notes; you'd have to read each one
individually. A NumPy array is the measuring tape: every element is the same
type, stored in a tight block of memory, so the computer can sweep over it in
one fast pass instead of reading each note.

**Illustration 2 — a blood pressure machine vs a paper chart.**
A digital blood pressure machine stores a column of numbers in a known format
and can compute your average in milliseconds. A paper chart can hold drawings,
tick marks, and comments, but the machine can't average a drawing. NumPy's
restriction (one type per array) is exactly what gives it its speed.

The technical version: a NumPy array is a contiguous block of memory where every
element has an identical size in bytes. The CPU can process this with vectorised
instructions (SIMD — single instruction, multiple data), applying one operation
to many elements in parallel. A Python list scatters its elements across memory
and adds a pointer to each, making vectorisation impossible.

## `np.array()` — turning a list into an array

The most direct way to create an array is to pass a Python list to `np.array()`.


In [ ]:
# A Python list of voltage readings (in millivolts, mV)
# Voltage = the electrical potential difference across a cell membrane.
# At rest, a neuron's inside is about -70 mV relative to the outside.
voltage_list = [-70.0, -68.5, -65.0, -55.0, 30.0, -75.0, -70.0]

# Convert to a NumPy array
voltage = np.array(voltage_list)

print("Array:", voltage)
print("Type:", type(voltage))          # numpy.ndarray (n-dimensional array)
print("Data type of elements:", voltage.dtype)  # float64 by default for floats
print("Shape:", voltage.shape)         # (7,) means 7 elements, 1 dimension
print("Number of elements:", voltage.size)


### The `dtype` parameter — choosing your element type

NumPy infers the element type automatically, but you can override it. The most
common types are:

| dtype | what it stores | typical use |
|-------|---------------|-------------|
| `float64` | 64-bit decimal | voltages, rates, time |
| `int32` / `int64` | whole numbers | spike counts, indices |
| `bool` | True / False | spike detection masks |

You choose `dtype` when memory matters (e.g. a million-element array of booleans
should not be stored as 64-bit floats) or when downstream code requires a specific
type.


In [ ]:
# Spike counts per trial — whole numbers, so int32 is appropriate
spike_counts = np.array([5, 12, 7, 0, 14, 9], dtype=np.int32)
print("Spike counts:", spike_counts)
print("dtype:", spike_counts.dtype)    # int32

# A boolean mask — which trials had any spikes at all?
had_spikes = np.array([True, True, True, False, True, True], dtype=bool)
print("Had spikes:", had_spikes)
print("dtype:", had_spikes.dtype)      # bool


## `np.zeros()`, `np.ones()`, `np.full()` — arrays built from scratch

Very often you need an array of a known size filled with a starting value, before
you fill it with real data in a loop or calculation. These three functions create
exactly that.


In [ ]:
# np.zeros(shape) — an array of zeros
# Think of it as a blank patient chart with labelled rows but no readings yet.
baseline = np.zeros(5)
print("Zeros:", baseline)              # [0. 0. 0. 0. 0.]  (float64 by default)

# A 2D zeros array: shape is a tuple (rows, columns)
# Think of it as a 3-neuron x 4-timepoint recording grid, all unread.
recording_grid = np.zeros((3, 4))
print("2D zeros:
", recording_grid)

# np.ones(shape) — all ones (useful as a starting multiplier or initial state)
ones = np.ones(4)
print("Ones:", ones)

# np.full(shape, fill_value) — fill with any constant
# E.g. initialise every reading to the resting potential -70 mV
resting = np.full(6, -70.0)
print("Resting potential array:", resting)


## `np.arange()` — evenly spaced integers (like `range()` but returns an array)

`range()` in Python produces a sequence you can loop over but can't do maths on
directly. `np.arange()` gives you the same sequence as a NumPy array, ready for
arithmetic.

**Illustration 1 — house numbers along a street.** If houses are numbered 10,
12, 14, 16 (start 10, step 2), `np.arange(10, 18, 2)` builds that list in one
call.

**Illustration 2 — a timer beeping every 5 milliseconds.** `np.arange(0, 1000, 5)`
gives you the beep times from 0 to 995 ms in one line.


In [ ]:
# np.arange(start, stop, step) — stop is EXCLUDED, like range()
# Build a time axis: 0, 1, 2, ..., 9 ms
time_ms = np.arange(0, 10, 1)
print("Time axis (ms):", time_ms)

# Step can be a float
time_fine = np.arange(0, 1.0, 0.1)   # 0.0, 0.1, 0.2, ..., 0.9
print("Fine time (s):", time_fine)

# Comparison: range() vs np.arange()
python_range = range(0, 10, 2)        # a lazy sequence, not an array
numpy_range  = np.arange(0, 10, 2)   # a real array you can do maths on
print("range type:", type(python_range))
print("arange type:", type(numpy_range))
print("arange * 2:", numpy_range * 2) # maths on the whole array at once
# python_range * 2 would REPEAT the sequence, not double each element!


## `np.linspace()` — evenly spaced floats including both endpoints

`linspace` means "linearly spaced." You tell it start, stop, and *how many
points*, and it divides the interval evenly. Crucially, the stop value **is
included** (unlike `arange`).

**Illustration 1 — dividing a road into equal segments.** You want to place
exactly 5 lampposts evenly between kilometre 0 and kilometre 10. `linspace(0,
10, 5)` places them at 0, 2.5, 5, 7.5, 10.

**Illustration 2 — a thermometer scale.** A thermometer labelled from 36 to 42
degrees with exactly 7 tick marks: `linspace(36, 42, 7)` gives ticks at every
whole degree.


In [ ]:
# np.linspace(start, stop, num) — num points from start to stop, inclusive
# Build a 1-second time axis with 1000 evenly spaced sample points (1 ms apart)
# This is how you'd create the time axis for a neural recording sampled at 1 kHz
t = np.linspace(0, 1.0, 1000)   # 0.000, 0.001, 0.002, ..., 1.000
print("First 5 time points (s):", t[:5])
print("Last 5 time points (s):", t[-5:])
print("Total points:", len(t))
print("Step between points:", t[1] - t[0])  # should be 0.001 s = 1 ms

# linspace vs arange for the same task
arr_arange   = np.arange(0, 1.001, 0.001)   # float steps can accumulate rounding
arr_linspace = np.linspace(0, 1.0, 1001)    # exact, no rounding accumulation
print("arange length:", len(arr_arange))     # might be 1000 or 1001 (rounding!)
print("linspace length:", len(arr_linspace)) # always exactly 1001
# Prefer linspace when the count matters; arange when the step size matters.


---
> ## Going deeper (optional on a first pass)
>
> Skip this block first time through. Come back when you want the full picture.
>
> **`np.empty(shape)`** allocates an array without initialising the values. It's
> fractionally faster than `zeros` when you know you will fill every cell before
> reading it. But read an uninitialised cell and you get whatever bytes were in
> that memory — could be anything. Use with care.
>
> **`np.eye(n)` and `np.identity(n)`** create the n x n identity matrix (1s on
> the diagonal, 0s elsewhere). You'll encounter it in linear algebra contexts.
>
> **`np.arange` and floating-point step accumulation.** Because binary floating
> point can't represent 0.1 exactly, repeated addition of a float step
> accumulates error. `np.arange(0, 1.0, 0.1)` sometimes returns 9 elements,
> sometimes 10, depending on rounding. This is why `linspace` is safer when you
> need a precise count.
>
> **`dtype` names in full.** `float64` = 64-bit double-precision IEEE 754 float
> (same as Python's default `float`). `float32` uses half the memory at some
> precision cost, useful for large data on GPUs. `int8` stores -128 to 127 only.
> `np.uint8` stores 0-255 (unsigned), the format of pixel values in images.
>
> **`np.frombuffer`, `np.fromfile`, `np.loadtxt`, `np.load`** load arrays
> directly from binary or text data — the real entry point for experimental data.
>
> **`np.logspace(start, stop, num)`** produces logarithmically spaced points,
> useful for frequency axes in signal processing (audio, EEG power spectra).

---

## Common questions and confusions

**"What is `ndarray`?"** Every NumPy array is technically an instance of
`numpy.ndarray` (n-dimensional array). `nd` signals it can be 1D, 2D, 3D, or
higher. You will see this name in error messages — don't be thrown by it.

**"Why does `np.zeros(5)` give floats? I wanted integers."** The default dtype
is `float64`. Add `dtype=int` to get integers: `np.zeros(5, dtype=int)`.

**"`arange` vs `linspace` — which should I use?"** Use `linspace` when you know
the number of points you need (and the exact endpoints matter). Use `arange` when
you know the step size. For float steps, always prefer `linspace` to avoid
rounding surprises.

**"Does `np.array([1, 2, 3])` copy the list or point to it?"** It always copies.
Changing the original list afterwards does not affect the array.

**"What does `shape` tell me?"** `shape` is a tuple: `(7,)` means 7 elements in
one dimension; `(3, 4)` means 3 rows and 4 columns. Number of elements in the
tuple = number of dimensions.

## Your exercises

Predict each answer **first**, then run and check.

1. Create `np.array([1, 2, 3, 4, 5])`. Print its `dtype` and `shape`. Now create
   the same from `[1.0, 2.0, 3.0, 4.0, 5.0]`. How does `dtype` differ, and why?
2. Create a 2D array of zeros with 4 rows and 3 columns using `np.zeros`. Print
   its `shape` and `size`. What is `size` for a (4, 3) array?
3. Build a time axis from 0 to 500 ms in steps of 1 ms using `np.arange`. How
   many elements does it have? Now build the same axis with `np.linspace` to get
   exactly 501 points. Are they the same?
4. Create `np.full((3, 3), -70.0)`. What does it look like? Change the fill value
   to `True` — what dtype does NumPy choose?
5. Predict: `np.arange(0, 1.0, 0.1)` — how many elements? Run it and count.
   Now try `np.linspace(0, 1.0, 11)`. Are the values the same?
6. *(Stretch.)* Create a 1D array of 100 evenly spaced angles from 0 to 2*pi
   (use `np.pi`). Then compute `np.sin` of every element in one line. What does
   the result look like — what kind of values do you expect?


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5
# your code here

# Exercise 6 (Stretch)
# your code here


## The irreducible core

1. A NumPy **array** stores elements of a **single type** in a contiguous block
   of memory, making maths across millions of elements fast.
2. Create from a list with **`np.array(list)`**; from scratch with **`np.zeros`**,
   **`np.ones`**, **`np.full`**.
3. **`np.arange(start, stop, step)`** -- stop excluded; use when step size is known.
4. **`np.linspace(start, stop, num)`** -- both endpoints included; use when count is
   known and precision matters.
5. Check an array's type with **`.dtype`**, its dimensions with **`.shape`**, and
   total element count with **`.size`**.

**You've got it when:** you can build a 1000-point time axis from 0 to 1 second
using both `arange` and `linspace` without looking anything up, and you can
explain why a NumPy array is faster than a Python list for numerical work.


---
# Item 2 — Indexing and Slicing Arrays

> **By the end of this you'll be able to:** pull individual elements out of 1D
> and 2D arrays, slice rows and columns, select elements by condition (Boolean
> indexing), and pick arbitrary elements by position (fancy indexing). Everything
> below the core sections is optional on a first pass.

## 1D indexing — same rules as Python lists

**Illustration 1 — a row of labelled test tubes.** A rack holds 8 tubes numbered
0 to 7 (scientists start at zero). "Give me tube 0" is the first; "give me tube
7" is the last; "give me tube -1" is also the last, counting from the right.

**Illustration 2 — a patient's medication history.** Seven days of records,
indexed Mon = 0 to Sun = 6. "Record -1" is Sunday (most recent). "Records 2 to 5"
is Wednesday through Friday (remember: stop index excluded).

The rules are exactly the same as lists, because NumPy arrays are designed to
feel familiar for 1D work.


In [ ]:
# A simulated voltage trace (in mV) — 8 time points
# Voltage trace: a record of a neuron's membrane voltage over time
voltage = np.array([-70.0, -68.0, -55.0, 10.0, 30.0, -10.0, -72.0, -70.0])

# Single element indexing (zero-based)
print("First reading:", voltage[0])    # -70.0  (index 0)
print("Peak voltage:", voltage[4])     # 30.0   (index 4)
print("Last reading:", voltage[-1])    # -70.0  (index -1 counts from end)
print("Second to last:", voltage[-2])  # -72.0

# Slicing: voltage[start:stop:step], stop is excluded
print("Indices 1 to 4:", voltage[1:5])         # 4 elements
print("Every other element:", voltage[::2])    # step=2
print("Reversed:", voltage[::-1])              # step=-1 reverses


## 2D arrays — shape, row/column indexing, slicing

A 2D array is a table: rows and columns. Think of it as a spreadsheet where every
cell holds one number.

**Illustration 1 — a spreadsheet of patient results.** Rows are patients, columns
are tests. To get patient 1's result on test 2, you specify row 1, column 2.

**Illustration 2 — a seating plan.** Rows A, B, C and seats 1, 2, 3. "Row B,
seat 2" gives you one chair. "All of row B" gives you the whole row. "Seat 2 in
every row" gives you a column.

In NumPy 2D indexing: `arr[row, col]`. Slices work in each dimension separately.


In [ ]:
# Neural data matrix: rows = neurons, columns = time points
# Each number is the firing rate (spikes per second, Hz) of that neuron at that time
# Firing rate = how many action potentials (spikes) a neuron fires per second
data = np.array([
    [2.0, 5.0, 8.0, 6.0, 3.0],   # neuron 0
    [0.0, 1.0, 12.0, 9.0, 0.0],  # neuron 1
    [4.0, 4.0, 4.0, 4.0, 4.0],   # neuron 2
])

print("Shape:", data.shape)        # (3 neurons, 5 time points)

# Single element: [row, col]
print("Neuron 1, time 2:", data[1, 2])   # 12.0

# Entire row (one neuron's full trace)
print("Neuron 0:", data[0, :])           # [2. 5. 8. 6. 3.]
print("Neuron 0 (shorthand):", data[0])  # same thing

# Entire column (all neurons at one time point)
print("Time point 2, all neurons:", data[:, 2])   # [8. 12.  4.]

# Sub-matrix: first two neurons, time points 1 to 3
print("Sub-matrix:
", data[0:2, 1:4])

# The colon : by itself means 'all of this dimension'
print("All neurons, time 0 only:", data[:, 0])    # [2. 0. 4.]


## Boolean indexing — filtering by condition

This is one of NumPy's most powerful features. When you write `arr[condition]`,
NumPy returns only the elements where the condition is `True`. The result is a
new, flat array of the matching values.

**Illustration 1 — a lab centrifuge alarm.** You have 50 sample tubes and a
machine that flags any tube above a temperature threshold. Boolean indexing is
exactly this: the condition (above threshold?) produces a True/False flag for
every tube, and you extract only the flagged ones.

**Illustration 2 — filtering a patient list by age.** "Give me all patients over
60" is a boolean filter: compare every age to 60, get True/False for each, select
the rows where True.


In [ ]:
# Simulate a 10-point voltage trace
v = np.array([-70, -65, -55, -40, 10, 30, 20, -60, -72, -70], dtype=float)

# Step 1: create the boolean mask (True where voltage > -50 mV)
# In neuroscience, -50 mV is near the 'threshold' — the voltage at which
# a neuron fires an action potential (a spike). Above threshold = likely spiking.
above_threshold = v > -50
print("Boolean mask:", above_threshold)   # [F F F T T T T F F F]

# Step 2: use the mask to extract the values
spike_voltages = v[above_threshold]
print("Voltages above threshold:", spike_voltages)

# You can write it in one line:
print("One-liner:", v[v > -50])

# Boolean indexing in 2D: select neurons (rows) with mean > 3 Hz
mean_rates = data.mean(axis=1)              # mean across columns (time), per neuron
print("Mean rates:", mean_rates)
active_neurons = data[mean_rates > 3]       # select rows where mean > 3
print("Active neuron rows:
", active_neurons)


## Fancy indexing — picking elements by a list of positions

Fancy indexing lets you pass a list (or array) of integer indices to select
multiple specific elements in any order.


In [ ]:
# Firing rates of 8 neurons
rates = np.array([1.2, 8.5, 0.3, 12.1, 4.7, 0.0, 7.9, 5.5])

# Select neurons 0, 2, 4 (every other starting from 0)
selected = rates[[0, 2, 4]]
print("Neurons 0, 2, 4:", selected)

# Select in a custom order (e.g. highest-responding neurons first)
top_indices = [3, 1, 6]        # we identified these as top responders
print("Top neurons:", rates[top_indices])

# Fancy indexing also works in 2D
# Select rows 0 and 2 (neurons 0 and 2) from our data matrix
print("Neurons 0 and 2:
", data[[0, 2]])


---
> ## Going deeper (optional on a first pass)
>
> **Views vs copies.** Basic slicing (e.g. `arr[1:4]`) returns a **view** — it
> shares memory with the original. Modify the slice and you modify the original.
> Boolean and fancy indexing always return a **copy**. This distinction is the
> source of many subtle bugs. Use `.copy()` when you want to guarantee
> independence: `chunk = arr[1:4].copy()`.
>
> **`np.where(condition, x, y)`** returns `x` where condition is True, `y` where
> False. It's a vectorised if/else over an entire array. Example:
> `np.where(v > -50, 1, 0)` replaces values with 1 (spike) or 0 (no spike).
>
> **`np.nonzero(condition)` and `np.argwhere(condition)`** return the indices
> where a condition is True, rather than the values. Useful when you need to know
> *which* time points had spikes, not just the voltages at those points.
>
> **Multidimensional fancy indexing.** You can index multiple dimensions
> simultaneously with arrays: `arr[[0, 1], [2, 3]]` selects elements (0,2) and
> (1,3) — not a sub-matrix, but specific cells.
>
> **Setting values with indexing.** Indexing on the left side of `=` sets values:
> `arr[arr < 0] = 0` clips all negatives to zero. This is how you write fast
> in-place operations without loops.

---

## Common questions and confusions

**"Is the stop index included or excluded?"** Always **excluded** in NumPy slices,
just like Python lists. `arr[1:4]` gives elements at indices 1, 2, 3.

**"My boolean mask has the wrong length."** The mask must have exactly as many
elements as the array dimension you're indexing. A common error is applying a
row-length mask to columns or vice versa.

**"Why does changing my slice change the original array?"** Basic slices return
views. If you don't want to affect the original, call `.copy()` on the slice.

**"What's the difference between `arr[0]` and `arr[0:1]`?"** `arr[0]` returns
a scalar (a plain number); `arr[0:1]` returns a 1-element array. They have
different shapes, which matters when you chain further operations.

**"Can I use boolean indexing to set values, not just select them?"**
Yes: `arr[arr < 0] = 0` sets every negative to zero in place. Very common and
very fast.

## Your exercises

Predict each answer **first**, then run and check.

1. Create `v = np.arange(10, 20)` (ten integers). Extract the element at index 3.
   What is it? Now extract the last element using negative indexing.
2. Slice `v` to get `[13, 14, 15, 16]`. Then slice to get every other element of
   the full array.
3. Create the 3x5 `data` matrix from the worked example. Extract column 3 (all
   neurons at time 3). Extract the sub-matrix covering neurons 0-1 and times 1-3.
4. Using `v` from exercise 1, create a boolean mask for values greater than 15.
   Print the mask, then use it to extract the matching values.
5. Using fancy indexing, extract elements at positions 0, 4, 9 from `v` (first,
   fifth, last). What is the result?
6. *(Stretch.)* In the `data` matrix, find which neurons (rows) have a value above
   10 at ANY time point. Use `np.any(data > 10, axis=1)` as your boolean mask to
   select those rows.


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5
# your code here

# Exercise 6 (Stretch)
# your code here


## The irreducible core

1. **1D indexing**: `arr[i]` for single element; `arr[start:stop:step]` for slices
   (stop excluded). Negative indices count from the end.
2. **2D indexing**: `arr[row, col]`; `:` means "all of this dimension."
3. **Boolean indexing**: `arr[arr > threshold]` — condition produces a True/False
   mask; only True elements are returned.
4. **Fancy indexing**: `arr[[i, j, k]]` — list of indices, any order, any
   repetition; always returns a copy.
5. Basic slices return **views** (shared memory); boolean and fancy indexing return
   **copies**. Use `.copy()` when independence matters.

**You've got it when:** given a (5, 10) neural data array, you can extract a
single neuron's trace, a time window across all neurons, and only the neurons
firing above a threshold — all without looking anything up.


---
# Item 3 — Array Operations and Broadcasting

> **By the end of this you'll be able to:** perform element-wise arithmetic on
> arrays, explain why this differs from list arithmetic, understand broadcasting
> (how NumPy handles arrays of different shapes), and use `np.dot()` for dot
> products. Everything below the core sections is optional on a first pass.

## Element-wise arithmetic — maths on entire arrays at once

**Illustration 1 — adjusting every patient's drug dosage by the same factor.**
A pharmacist has 50 patients on a drug at 10 mg each. A new protocol doubles
the dose. Instead of writing 50 new prescriptions one at a time, you apply the
multiplier to the whole list at once: `doses * 2`. That is element-wise
multiplication.

**Illustration 2 — converting a thermometer reading from Celsius to Fahrenheit.**
You have 1000 temperature readings. The formula `F = C * 9/5 + 32` needs to be
applied to every reading. NumPy does this to all 1000 in a single expression.

With plain Python lists, `list + list` **concatenates** (joins end to end). With
NumPy arrays, `arr + arr` **adds element by element**. This is the single most
common source of confusion when switching from lists to arrays.


In [ ]:
# The critical difference between list and array arithmetic
py_list = [1, 2, 3]
np_arr  = np.array([1, 2, 3])

print("list + list:", py_list + py_list)   # [1, 2, 3, 1, 2, 3] concatenation!
print("arr + arr:", np_arr + np_arr)        # [2, 4, 6] element-wise addition

# A simulated voltage trace (mV) over 6 time points
# Resting potential: the baseline voltage of a neuron when not actively firing
resting = -70.0
voltage_trace = np.array([-70.0, -68.0, -55.0, 10.0, 30.0, -72.0])

# Subtract the resting potential from every point (centres the trace on 0)
# This is a very common preprocessing step in electrophysiology
centred = voltage_trace - resting
print("Centred trace:", centred)    # each value relative to -70 mV

# Scale the trace (e.g. convert from mV to V: divide every element by 1000)
voltage_V = voltage_trace / 1000.0
print("In volts:", voltage_V)

# Add noise to every element (simulating measurement noise)
noise_level = 0.5  # mV
noisy = voltage_trace + noise_level
print("With noise:", noisy)

# All four standard arithmetic operations are element-wise
a = np.array([10.0, 20.0, 30.0])
b = np.array([2.0, 4.0, 6.0])
print("a + b:", a + b)   # [12. 24. 36.]
print("a - b:", a - b)   # [ 8. 16. 24.]
print("a * b:", a * b)   # [20. 80. 180.]
print("a / b:", a / b)   # [5. 5. 5.]


## Broadcasting — when shapes don't match exactly

NumPy can operate on arrays of different shapes, as long as they are compatible.
This is called **broadcasting**: the smaller array is conceptually "stretched" to
match the larger one.

**The rules (simplified for the cases you'll actually meet):**
- A scalar (single number) broadcasts over any array: `arr * 3` multiplies every
  element by 3.
- A 1D array of length N broadcasts over a 2D array of shape (M, N): it's applied
  to each row.
- Two arrays broadcast if their shapes match from the right, with 1s allowed to
  stretch.

**Illustration 1 — a single calibration factor applied to every instrument.**
A lab has 5 instruments, each recording 100 values. You discover a calibration
offset that applies to every instrument. Broadcasting lets you add a single
offset value to a (5, 100) array — no loop needed.

**Illustration 2 — scaling each student's score by a different factor.**
Imagine a (30 students, 5 tests) score matrix. You want to apply a different
weight to each test (a 5-element array). Broadcasting applies the weight vector
to every row simultaneously.


In [ ]:
# Scalar broadcasting: the simplest case
firing_rates = np.array([2.0, 8.0, 5.0, 0.0, 11.0])

# Normalise: divide every rate by the maximum to get values between 0 and 1
max_rate = firing_rates.max()
normalised = firing_rates / max_rate   # scalar max_rate broadcasts over the array
print("Normalised rates:", normalised)

# 2D broadcasting: apply a per-neuron baseline to a matrix
# data has shape (3, 5) — 3 neurons, 5 time points
data = np.array([
    [2.0, 5.0, 8.0, 6.0, 3.0],
    [0.0, 1.0, 12.0, 9.0, 0.0],
    [4.0, 4.0, 4.0, 4.0, 4.0],
])

# Baseline rate for each neuron (a 1D array of length 3)
baseline = np.array([1.0, 0.5, 2.0])

# To subtract each neuron's own baseline, reshape baseline to (3, 1)
# so it broadcasts down each row: (3,1) broadcasts with (3,5) -> (3,5)
baseline_col = baseline.reshape(3, 1)  # turn row vector into column vector
data_centred = data - baseline_col
print("Data after subtracting per-neuron baseline:
", data_centred)

# Simple row-wise broadcasting: apply a time-axis multiplier to every neuron
# time_weights shape (5,) broadcasts with data shape (3, 5) along columns
time_weights = np.array([0.5, 0.75, 1.0, 0.75, 0.5])  # taper edges
tapered = data * time_weights
print("Tapered data:
", tapered)


## `np.dot()` — the dot product

The dot product of two vectors is the sum of their element-wise products. It
appears constantly in neuroscience: weighted sums of inputs, projections,
correlations.

For two 1D arrays of length N: `np.dot(a, b)` = a[0]*b[0] + a[1]*b[1] + ... +
a[N-1]*b[N-1]. The result is a single number.

For 2D arrays (matrices), `np.dot(A, B)` is matrix multiplication. This will
become important in the SciPy notebook. For now, just know the 1D case.


In [ ]:
# Dot product example: a simple neural model
# A neuron receives input from 4 upstream neurons.
# inputs: the activation level (arbitrary units) of each upstream neuron
# weights: how strongly each input drives our neuron (positive = excitatory,
#          negative = inhibitory; excitatory means 'tends to make it fire',
#          inhibitory means 'tends to suppress firing')
inputs  = np.array([1.0, 0.5, 0.8, 0.2])
weights = np.array([0.9, -0.5, 0.3, 1.2])

# The neuron sums up weighted inputs — the dot product
weighted_sum = np.dot(inputs, weights)
print("Weighted sum of inputs:", weighted_sum)

# Manually, to check: same result
manual = np.sum(inputs * weights)
print("Manual check:", manual)

# Note: for 1D arrays, @ operator is equivalent to np.dot
print("Using @ operator:", inputs @ weights)


---
> ## Going deeper (optional on a first pass)
>
> **Universal functions (ufuncs).** All NumPy arithmetic operators (`+`, `-`, `*`,
> `/`) are wrappers around C-level functions that operate element-wise without a
> Python loop. They are called **universal functions** or ufuncs. Examples:
> `np.add`, `np.multiply`, `np.sqrt`, `np.exp`, `np.log`. You can call them as
> functions too: `np.sqrt(arr)` applies the square root to every element.
>
> **Broadcasting rules in full.** NumPy pads shapes on the **left** with 1s until
> both arrays have the same number of dimensions. Then dimensions are compatible if
> they are equal OR one of them is 1. A dimension of size 1 is stretched to match.
> Incompatible shapes raise `ValueError: operands could not be broadcast together`.
>
> **`@` vs `np.dot` vs `np.matmul`.** For 1D arrays, all three give the dot
> product. For 2D, `np.dot` and `@` differ for higher-dimensional arrays: `@`
> (via `np.matmul`) treats them as stacks of matrices; `np.dot` does a sum over
> the last axis of the first and second-to-last of the second. For strictly 2D
> work, `@` is clearest.
>
> **In-place operations.** `arr += 1` modifies `arr` in place, avoiding a new
> allocation. Useful in tight loops for memory efficiency, but only works when the
> dtype can hold the result.
>
> **`np.outer(a, b)`** computes the outer product (a column times a row = a
> matrix). It appears in Hebbian learning rules in neural models.

---

## Common questions and confusions

**"`list + list` gave me a longer list, not added values. What happened?"**
For Python lists, `+` means concatenate (join together). For NumPy arrays, `+`
means element-wise addition. This is the single most common beginner mistake when
mixing the two. Always know which type you're working with.

**"I got a broadcasting error. What do I do?"** Print the `.shape` of both arrays.
Shapes must match from the right; mismatches show up there. The fix is usually a
`.reshape()` to make one dimension equal 1 so it can stretch.

**"Does `arr * arr` give me the dot product?"** No. `arr * arr` is element-wise
multiplication (squares each element). For the dot product (sum of products), use
`np.dot(arr, arr)` or `arr @ arr`.

**"When does broadcasting happen silently vs raising an error?"** Shapes are
compatible if they match from the right with 1s allowed to stretch. Any other
mismatch raises an error. Example: (3, 5) and (3,) are incompatible (5 != 3 from
right); (3, 5) and (5,) are compatible.

## Your exercises

Predict each answer **first**, then run and check.

1. Create `a = np.array([1, 2, 3, 4])` and `b = np.array([10, 20, 30, 40])`.
   Compute `a + b`, `b - a`, `a * b`, and `b / a`. Verify the results mentally.
2. Create `v = np.array([-70., -65., -55., 10., 30., -72.])`. Subtract `−70.0`
   from every element (remove the resting potential). What does the result mean
   biologically?
3. Create a (3, 4) array of ones. Try to add a 1D array of length 4 to it —
   predict whether broadcasting will succeed and what the result looks like.
   Then try adding a length-3 array. What happens?
4. Compute `np.dot(np.array([1., 0., -1.]), np.array([3., 5., 2.]))` by hand
   first, then verify with NumPy.
5. Given `rates = np.array([2., 5., 8., 3., 11.])`, normalise it so the maximum
   is 1 and the minimum is 0. (Hint: `(rates - rates.min()) / (rates.max() -
   rates.min())`.)
6. *(Stretch.)* Create a (4, 5) matrix where each row is `[0, 1, 2, 3, 4]`
   multiplied by the row index (row 0 all zeros, row 1 is [0,1,2,3,4], row 2 is
   [0,2,4,6,8], etc.). Use broadcasting with `np.arange` and `.reshape`. Then
   compute the dot product of row 1 and row 3.


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5
# your code here

# Exercise 6 (Stretch)
# your code here


## The irreducible core

1. NumPy arithmetic (`+`, `-`, `*`, `/`) is **element-wise** — it operates on
   paired elements. Python list `+` is concatenation; NumPy array `+` is addition.
2. A **scalar** always broadcasts over an array: `arr * 3` multiplies every element.
3. **Broadcasting** lets arrays of compatible shapes operate together; shapes match
   from the right, with 1s stretching to fit.
4. **`np.dot(a, b)`** (or `a @ b`) computes the dot product: sum of element-wise
   products. For matrices it is matrix multiplication.
5. When a broadcasting error occurs, print `.shape` on both arrays and compare
   from the right.

**You've got it when:** you can subtract a per-neuron baseline from a (neurons,
time) matrix using broadcasting without a loop, and explain why `list + list`
and `array + array` behave differently.


---
# Item 4 — Aggregation and Statistical Functions

> **By the end of this you'll be able to:** compute summary statistics (sum,
> mean, std, min, max, median) across an entire array or along rows/columns,
> find the index of extreme values with argmin/argmax, and compute cumulative
> sums and differences. Everything below the core sections is optional on a
> first pass.

## Aggregation functions — collapsing many numbers to one

**Illustration 1 — averaging test scores across a class.** You have 30 students,
each with a score. `np.mean(scores)` computes the class average in one call,
doing what you would otherwise loop over manually.

**Illustration 2 — column and row totals in a spreadsheet.** A spreadsheet's
SUM at the bottom of a column or end of a row is exactly `np.sum(data, axis=0)`
(column totals) or `np.sum(data, axis=1)` (row totals).

The key functions all work on a whole array by default, collapsing everything to
a single number.


In [ ]:
# Simulated spike counts: how many spikes each of 8 neurons fired in one trial
# Spike = action potential: the brief electrical pulse a neuron sends when it fires
spike_counts = np.array([5, 12, 7, 0, 14, 9, 3, 8])

print("Sum of all spikes:", np.sum(spike_counts))       # total spikes
print("Mean spike count:", np.mean(spike_counts))       # average across neurons
print("Std dev:", np.std(spike_counts))                 # how much counts vary
print("Minimum:", np.min(spike_counts))                 # quietest neuron
print("Maximum:", np.max(spike_counts))                 # most active neuron
print("Median:", np.median(spike_counts))               # middle value

# Methods: arrays also carry these as built-in methods (.mean(), .sum(), etc.)
print("Via method:", spike_counts.mean())               # same as np.mean()


## The `axis` parameter — collapsing along rows or columns

When you have a 2D array, you often want to summarise along one dimension
rather than collapsing everything. The `axis` parameter controls which dimension
is collapsed.

- `axis=0` collapses **rows** (operates down each column). Result has the shape
  of one row.
- `axis=1` collapses **columns** (operates across each row). Result has the shape
  of one column (returned as a 1D array).

**Memory trick:** `axis=0` removes the rows (the "0" dimension); `axis=1` removes
the columns (the "1" dimension). Whatever you specify gets squashed away.


In [ ]:
# Neural data: rows = neurons (3), columns = time points (5)
data = np.array([
    [2.0, 5.0, 8.0, 6.0, 3.0],   # neuron 0
    [0.0, 1.0, 12.0, 9.0, 0.0],  # neuron 1
    [4.0, 4.0, 4.0, 4.0, 4.0],   # neuron 2
])
print("Shape:", data.shape)   # (3, 5)

# No axis: collapse everything to one number
print("Overall mean:", np.mean(data))          # mean of all 15 values

# axis=0: mean down each column (across neurons, one value per time point)
# "What was the population average activity at each time point?"
mean_over_neurons = np.mean(data, axis=0)
print("Mean across neurons (per time):", mean_over_neurons)  # shape (5,)

# axis=1: mean across each row (across time, one value per neuron)
# "What was each neuron's average activity over the whole trial?"
mean_over_time = np.mean(data, axis=1)
print("Mean across time (per neuron):", mean_over_time)      # shape (3,)

# Standard deviation per neuron (how variable was each neuron?)
std_per_neuron = np.std(data, axis=1)
print("Std per neuron:", std_per_neuron)

# Sum over time for each neuron (total spikes in a rate-coded model)
total_per_neuron = np.sum(data, axis=1)
print("Total activity per neuron:", total_per_neuron)


## `np.argmin()` and `np.argmax()` — finding the index of extreme values

Sometimes you don't want the maximum value — you want to know *where* in the
array it occurs. `argmax` and `argmin` return the **index** of the extreme value,
not the value itself.

**Illustration:** finding the time at which a patient's temperature peaked.
`np.argmax(temperature)` gives you the index (time step), not the peak value.


In [ ]:
# A simulated voltage trace: find the action potential peak
# Action potential: the rapid rise and fall of voltage when a neuron fires a spike
v = np.array([-70.0, -68.0, -55.0, -20.0, 30.0, 10.0, -40.0, -72.0, -70.0])

peak_index = np.argmax(v)          # index of the maximum value
peak_value = v[peak_index]         # the value at that index
print("Peak voltage:", peak_value, "mV at time index", peak_index)

# You can also use: v.argmax()
trough_index = np.argmin(v)
print("Trough (most negative):", v[trough_index], "mV at index", trough_index)

# In 2D: argmax along axis=1 gives the time of peak for each neuron
peak_times = np.argmax(data, axis=1)
print("Time of peak activity per neuron:", peak_times)  # index of max in each row


## `np.cumsum()` and `np.diff()` — running totals and consecutive differences

**`np.cumsum()`** gives the running total: element i of the result is the sum of
all elements up to and including position i.

**`np.diff()`** gives the difference between consecutive elements: element i of
the result is `arr[i+1] - arr[i]`. The output is one element shorter than the
input.

In neuroscience, `np.diff()` applied to an array of spike times gives the
**inter-spike intervals (ISIs)** — the gaps between successive spikes. ISI
distributions reveal important properties of how neurons encode information.


In [ ]:
# np.cumsum: running total of spike counts across trials
spike_counts_trials = np.array([5, 12, 7, 0, 14, 9])
running_total = np.cumsum(spike_counts_trials)
print("Running total:", running_total)   # [5, 17, 24, 24, 38, 47]

# np.diff: differences between consecutive elements
# Spike times (in ms) for a single neuron during a recording
# A spike train is the sequence of time points at which a neuron fires
spike_times = np.array([10, 25, 42, 43, 80, 120, 125, 200])  # ms

# Inter-spike intervals (ISI): time between consecutive spikes
# Short ISIs = rapid firing; long ISIs = silent periods
isi = np.diff(spike_times)
print("Spike times:", spike_times)
print("Inter-spike intervals (ms):", isi)   # 8 spikes -> 7 intervals
print("Mean ISI:", np.mean(isi), "ms")
print("Min ISI:", np.min(isi), "ms  (refractory period?)")
# The refractory period is the brief time after a spike during which a neuron
# cannot fire again — typically 1-2 ms

# diff on a voltage trace gives the rate of voltage change (dV/dt approximation)
dv = np.diff(v)
print("dV between samples:", dv)


---
> ## Going deeper (optional on a first pass)
>
> **`ddof` parameter in `np.std` and `np.var`.** By default, NumPy divides by N
> (population std). For the unbiased sample standard deviation, use `ddof=1`
> (divides by N-1). Most statistics courses teach the N-1 version. In
> neuroscience data analysis you usually want `ddof=1` when estimating population
> variability from a sample.
>
> **`np.percentile(arr, q)` and `np.quantile(arr, q/100)`.** Find the value below
> which q% of the data falls. `np.percentile(arr, 50)` equals the median.
>
> **`np.histogram(arr, bins)`.** Computes a histogram (counts per bin) without
> plotting. Returns `(counts, bin_edges)`. Useful for analysing ISI distributions
> or firing rate histograms.
>
> **`keepdims=True`.** When reducing along an axis, the result normally drops that
> dimension. `np.mean(data, axis=1, keepdims=True)` keeps the shape as (3, 1)
> rather than (3,), which makes subsequent broadcasting cleaner.
>
> **`np.nanmean`, `np.nanstd`, `np.nansum`.** Variants that ignore `NaN` (Not a
> Number) values, which appear in real data wherever measurements are missing or
> invalid. The plain versions propagate NaN through the calculation, often ruining
> results.

---

## Common questions and confusions

**"Which axis is rows and which is columns?"** For a (rows, cols) array: `axis=0`
is the rows axis (collapses rows, leaves columns); `axis=1` is the columns axis
(collapses columns, leaves rows). When unsure, check the shape of the output.

**"Why does `np.diff()` give me one fewer element?"** Because differences are
*between* elements: N elements produce N-1 gaps. If you have 8 spike times, you
have 7 inter-spike intervals.

**"Is `np.std()` the population or sample standard deviation?"** Population by
default (divides by N). Use `ddof=1` for the sample version (divides by N-1).

**"`np.argmax()` only gives me one index. What if there are ties?"** It returns
the index of the first maximum. If there are multiple equal maxima, only the first
is returned. `np.where(arr == arr.max())` gives all of them.

**"Can I use these functions on lists directly?"** Yes, `np.sum([1,2,3])` works.
NumPy converts the list internally. But the result is a NumPy scalar, not a
Python int.

## Your exercises

Predict each answer **first**, then run and check.

1. Create `scores = np.array([88, 72, 95, 61, 79, 84, 91])`. Compute the mean,
   median, min, max, and standard deviation. Which neuron (index) has the maximum
   score?
2. Create a (4, 3) matrix of your choice. Compute the sum along `axis=0` and
   `axis=1`. Check that their totals both equal `np.sum(matrix)`.
3. Given `spike_times = np.array([5, 18, 34, 50, 51, 90])`, compute the ISIs with
   `np.diff`. What is the minimum ISI? What does that tell you about the neuron?
4. Compute `np.cumsum(np.array([1, 1, 1, 1, 1]))`. What do you get and why?
5. For the `data` matrix from the worked examples, find the time index (column)
   of peak activity for each neuron using `np.argmax(data, axis=1)`.
6. *(Stretch.)* For the `data` matrix, normalise each neuron's trace by subtracting
   its mean and dividing by its standard deviation (z-score normalisation). Use
   `axis=1` operations and broadcasting. After normalisation, what should the mean
   and std of each row be?


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5
# your code here

# Exercise 6 (Stretch)
# your code here


## The irreducible core

1. **`np.sum`, `np.mean`, `np.std`, `np.min`, `np.max`, `np.median`** collapse an
   array to summary statistics.
2. The **`axis` parameter** controls which dimension is collapsed: `axis=0` removes
   the rows axis; `axis=1` removes the columns axis.
3. **`np.argmin` / `np.argmax`** return the **index** of the extreme value, not
   the value itself.
4. **`np.cumsum`** gives the running total; **`np.diff`** gives consecutive
   differences (result is one element shorter).
5. For sample statistics (not whole-population), use **`ddof=1`** with `np.std`
   and `np.var`.

**You've got it when:** given a (neurons x time) matrix, you can compute the mean
firing rate per neuron, find which neuron peaked latest using `argmax`, and
compute ISIs from a spike-time array — all without a loop.


---
# Item 5 — Random Numbers with numpy.random

> **By the end of this you'll be able to:** generate reproducible random numbers,
> sample from uniform and normal distributions, use `randint` and `choice`,
> and simulate a noisy voltage trace and a Poisson spike train. Everything below
> the core sections is optional on a first pass.

## Why random numbers matter in science

Simulations, statistical tests, shuffling, sampling, model initialisation, noise
injection — all require random numbers. NumPy's random module gives you
statistical quality pseudorandom numbers fast.

## `np.random.seed()` — making randomness reproducible

**Illustration 1 — dice rolls.** If you could secretly set a hidden starting
configuration of the dice before a roll, you'd get the same sequence every time
you used that configuration. A seed is that hidden starting configuration.

**Illustration 2 — drawing names from a hat.** If the hat is prepared the same
way (same order of names, same starting position), drawing produces the same
sequence every time. The seed is the preparation.

A random number generator is not truly random — it's a deterministic algorithm
starting from a seed number and producing sequences that *look* random. Same seed,
same sequence. This is **pseudorandom** generation, and for science it's good:
you can share code that reproduces results exactly.


In [ ]:
# Without a seed, results change every run
print("Without seed:", np.random.rand(3))
print("Again:", np.random.rand(3))    # different!

# With a fixed seed, results are reproducible
np.random.seed(42)
print("Seed 42, first call:", np.random.rand(3))
np.random.seed(42)                    # re-set to same seed
print("Seed 42, again:", np.random.rand(3))  # identical to first call

# Best practice: set the seed once at the top of your analysis script


## `np.random.rand()` and `np.random.randn()` — uniform vs normal

**`rand()`** draws from a **uniform** distribution between 0 and 1. Every value
in [0, 1) is equally likely. Think of it as a spinner that stops anywhere on a
circle with equal probability.

**`randn()`** draws from a **standard normal** (Gaussian) distribution: mean 0,
standard deviation 1. Most values cluster near 0; extremes are rare. Think of
height measurements across a population — most people are near the average, very
few are extremely tall or short.


In [ ]:
np.random.seed(0)

# rand(): uniform [0, 1)
uniform_samples = np.random.rand(5)
print("Uniform [0,1):", uniform_samples)

# rand with shape: pass shape as separate arguments
uniform_2d = np.random.rand(3, 4)   # 3x4 matrix of uniform values
print("2D uniform shape:", uniform_2d.shape)

# randn(): standard normal (mean=0, std=1)
normal_samples = np.random.randn(5)
print("Standard normal:", normal_samples)   # mostly between -2 and 2

# Scaling randn() to any normal distribution
# A neuron's resting membrane potential fluctuates slightly due to ion channel noise
# resting potential = -70 mV, noise std = 2 mV
mean_v = -70.0
std_v  = 2.0
noisy_resting = mean_v + std_v * np.random.randn(1000)
print("Noisy resting potential -- mean:", noisy_resting.mean().round(2),
      "std:", noisy_resting.std().round(2))


## `np.random.normal()`, `np.random.poisson()` — named distributions

These functions let you sample directly from named distributions with explicit
parameters, which is clearer than the manual scaling of `randn()`.

**Normal distribution:** the classic bell curve. Parameterised by mean and
standard deviation.

**Poisson distribution:** the distribution of counts when events occur randomly
at a constant average rate. A Poisson spike train is the standard model of how
a neuron fires: each small time window, a spike occurs with a small fixed
probability, giving Poisson-distributed spike counts.


In [ ]:
np.random.seed(1)

# np.random.normal(mean, std, size) — explicitly parameterised normal
# Simulate 200 ms of a neuron's resting potential sampled every 1 ms
resting_trace = np.random.normal(loc=-70.0, scale=3.0, size=200)
# loc = mean, scale = standard deviation, size = number of samples
print("Trace shape:", resting_trace.shape)
print("Mean:", resting_trace.mean().round(2), "Std:", resting_trace.std().round(2))

# np.random.poisson(lam, size) — Poisson spike counts
# lam = average number of events (spikes) per time unit
# If a neuron fires at 10 spikes/second (Hz), and we record 100 trials of 1 second,
# each trial's spike count should average ~10
mean_spike_rate = 10  # spikes per second
n_trials = 100
spike_count_trials = np.random.poisson(lam=mean_spike_rate, size=n_trials)
print("Poisson spike counts, first 10 trials:", spike_count_trials[:10])
print("Sample mean:", spike_count_trials.mean().round(2))   # should be ~10
print("Sample std:", spike_count_trials.std().round(2))     # should be ~sqrt(10) ~ 3.16
# A key property of the Poisson distribution: variance = mean (so std = sqrt(mean))


## `np.random.randint()` and `np.random.choice()`

**`randint(low, high, size)`** draws integers uniformly from [low, high) — just
like rolling dice.

**`choice(a, size, replace)`** draws samples from an array or range, with or
without replacement — like drawing names from a hat.


In [ ]:
np.random.seed(7)

# randint: simulate 20 dice rolls (values 1-6)
dice_rolls = np.random.randint(low=1, high=7, size=20)  # high is excluded, so 7 gives 1-6
print("Dice rolls:", dice_rolls)

# In a neuroscience context: randomly assign 30 subjects to 3 conditions (0, 1, 2)
n_subjects = 30
condition_labels = np.random.randint(0, 3, size=n_subjects)
print("Condition assignments:", condition_labels)

# choice: draw from a specific array
neuron_ids = np.arange(100)   # we have 100 neurons

# Pick 10 neurons at random WITHOUT replacement (each chosen at most once)
selected_neurons = np.random.choice(neuron_ids, size=10, replace=False)
print("Selected neurons:", selected_neurons)

# choice can also take probabilities — e.g. neurons with unequal sampling likelihood
# (useful for modelling non-uniform stimulus presentation)
stimuli = np.array(["grating", "dots", "noise"])
probs   = np.array([0.5, 0.3, 0.2])   # must sum to 1
n_trials_stim = 20
trial_stimuli = np.random.choice(stimuli, size=n_trials_stim, p=probs)
print("Trial stimuli:", trial_stimuli)


## Putting it together — simulating a noisy membrane potential and spike detection

Let's combine random number generation with the array skills from earlier items
to simulate a realistic (simple) neuron recording.


In [ ]:
np.random.seed(2024)

# Parameters
dt       = 0.001       # time step: 1 ms = 0.001 s
duration = 0.5         # 500 ms recording
t        = np.linspace(0, duration, int(duration / dt))   # time axis

# Resting membrane potential with Gaussian noise
resting   = -70.0      # mV
noise_std = 5.0        # mV (biological noise from ion channels)
v_noise   = np.random.normal(resting, noise_std, size=len(t))

# Simulate a brief current injection at t=0.1 s that drives voltage up
# Find the indices corresponding to 100-200 ms
stim_start = int(0.10 / dt)
stim_end   = int(0.20 / dt)
v_noise[stim_start:stim_end] += 20.0   # add 20 mV during stimulation window

# Detect 'spikes': time points where voltage crosses -50 mV (a simple threshold)
threshold = -50.0
above = v_noise > threshold
spike_indices = np.where(above)[0]  # indices where voltage crossed threshold
spike_times_ms = t[spike_indices] * 1000  # convert to ms

print("Total time points:", len(t))
print("Number of threshold crossings:", len(spike_indices))
print("Spike-like times (ms):", spike_times_ms[:5], "...")
print("Mean voltage during stim:", v_noise[stim_start:stim_end].mean().round(2), "mV")


---
> ## Going deeper (optional on a first pass)
>
> **The new random API (`numpy.random.default_rng`).** NumPy 1.17 introduced a
> cleaner interface: `rng = np.random.default_rng(seed=42)` creates a Generator
> object. Use `rng.normal(...)`, `rng.integers(...)`, etc. This is now the
> recommended approach for new code; it is faster and offers more distributions.
> The old `np.random.seed` / `np.random.rand` API still works but is considered
> legacy.
>
> **Reproducibility in parallel code.** When multiple threads or processes each
> call `np.random.rand()`, they share a global state and the sequence is
> unpredictable. Each process should create its own Generator with its own seed.
>
> **Other distributions.** `np.random.exponential(scale, size)` gives
> exponentially distributed ISIs (the ISI distribution for a Poisson spike train
> is exactly exponential). `np.random.uniform(low, high, size)` is explicit
> uniform. `np.random.binomial(n, p, size)` counts successes in n Bernoulli
> trials.
>
> **Shuffling.** `np.random.shuffle(arr)` shuffles an array in place.
> `np.random.permutation(n)` returns a new permuted array (or shuffled range).
> Used for permutation tests in statistics.
>
> **Why "pseudorandom"?** True randomness requires a physical source (thermal
> noise, radioactive decay). Computers are deterministic, so they use algorithms
> (Mersenne Twister in the old API, PCG64 in the new API) that produce sequences
> that pass statistical randomness tests. For scientific purposes they are
> indistinguishable from true random.

---

## Common questions and confusions

**"What is the difference between `rand()` and `randn()`?"** `rand()` gives
uniform values in [0, 1); `randn()` gives normally distributed values centred on
0 with std 1. They are easy to confuse by name; remember `n` = normal.

**"How do I get a normal distribution with a specific mean and std?"** Two ways:
`mean + std * np.random.randn(n)` or `np.random.normal(mean, std, n)`. The second
is clearer.

**"Does setting the seed at the top of a cell in Colab guarantee reproducibility?"**
Only if the cell is run fresh from that seed. If you call other random functions
first (including in other cells), the sequence is advanced. For safety, set the
seed immediately before the code you want to reproduce.

**"Is `np.random.randint(0, 10)` inclusive of 10?"** No. The high value is
excluded, matching Python's `range()`. `randint(0, 10)` gives 0 to 9.

**"When should I use `replace=False` in `choice`?"** When sampling without
replacement (each item chosen at most once) -- as in drawing names from a hat or
selecting a random subset of neurons for further analysis without repeats.

## Your exercises

Predict each answer **first**, then run and check.

1. Set seed 99 and generate 5 values with `np.random.rand(5)`. Describe the range
   of values. Reset seed 99 and generate again -- are they the same?
2. Generate 1000 values from `np.random.randn(1000)`. Compute their mean and std.
   What values do you expect? How close are the sample statistics?
3. Simulate 50 trials of a neuron firing at a mean rate of 5 Hz using
   `np.random.poisson`. Compute the sample mean and std and compare them to the
   theoretical values (mean = 5, std = sqrt(5)).
4. Generate 10 random integers in [0, 100) with `randint`. Find the maximum and
   the index of the maximum.
5. You have 20 participants. Randomly assign them to two equal groups (10 each)
   using `np.random.choice` without replacement. (Hint: choose 10 indices from
   range(20); the rest are the other group.)
6. *(Stretch.)* Simulate a 1-second Poisson spike train at 20 Hz sampled at 1 kHz
   (1000 time bins). Each time bin has probability 20/1000 = 0.02 of containing a
   spike. Use `np.random.rand(1000) < 0.02` to generate a boolean spike train.
   Count the total spikes and compute the firing rate. How close to 20 Hz?


In [ ]:
# Exercise 1
# your code here

# Exercise 2
# your code here

# Exercise 3
# your code here

# Exercise 4
# your code here

# Exercise 5
# your code here

# Exercise 6 (Stretch)
# your code here


## The irreducible core

1. **`np.random.seed(n)`** fixes the starting state so the same sequence is
   reproduced every run. Always set it for scientific reproducibility.
2. **`rand()`** = uniform [0, 1); **`randn()`** = standard normal (mean 0, std 1).
   Scale randn: `mean + std * randn(n)`.
3. **`np.random.normal(mean, std, size)`** and **`np.random.poisson(lam, size)`**
   sample explicitly from named distributions.
4. **`randint(low, high, size)`** gives random integers; `high` is excluded.
   **`choice(a, size, replace=...)`** draws from an array, with or without
   replacement.
5. The new API **`np.random.default_rng(seed)`** is cleaner for new code; the old
   `np.random.seed` interface still works.

**You've got it when:** you can write a 10-line script that sets a seed, generates
a noisy voltage trace with `normal`, detects threshold crossings, and reports the
firing rate -- and get the same numbers every run.


---
# Solutions -- try first!

Work through every exercise yourself before looking here. Peeking early short-
circuits the learning. The gap between your prediction and the actual output is
where real understanding forms.


## Item 1 Solutions — Creating Arrays

In [ ]:
# Exercise 1: dtype from integers vs floats
arr_int   = np.array([1, 2, 3, 4, 5])
arr_float = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
print("Int array dtype:", arr_int.dtype, "  shape:", arr_int.shape)
print("Float array dtype:", arr_float.dtype, " shape:", arr_float.shape)
# The decimal point in 1.0 signals a float; 1 without decimal is int


In [ ]:
# Exercise 2: zeros shape and size
z = np.zeros((4, 3))
print("Shape:", z.shape)   # (4, 3)
print("Size:", z.size)     # 4 * 3 = 12


In [ ]:
# Exercise 3: arange vs linspace for 0-500 ms
arange_axis  = np.arange(0, 501, 1)    # 0 to 500 inclusive with step 1
linspace_axis = np.linspace(0, 500, 501)
print("arange length:", len(arange_axis))      # 501
print("linspace length:", len(linspace_axis))  # 501
print("Values match:", np.allclose(arange_axis, linspace_axis))  # True


In [ ]:
# Exercise 4: np.full with different fill values
full_float = np.full((3, 3), -70.0)
print(full_float)
print("dtype:", full_float.dtype)  # float64

full_bool = np.full((3, 3), True)
print("Bool fill dtype:", full_bool.dtype)  # bool


In [ ]:
# Exercise 5: arange with float step
a = np.arange(0, 1.0, 0.1)
print("arange length:", len(a), " values:", a)
# May be 10 elements (due to float rounding, 1.0 is excluded)

b = np.linspace(0, 1.0, 11)
print("linspace length:", len(b), " values:", b)
# Always exactly 11 points; values very close but not identical due to rounding


In [ ]:
# Exercise 6 (Stretch): angles and sine
angles = np.linspace(0, 2 * np.pi, 100)
sines  = np.sin(angles)
print("First 5 sines:", sines[:5].round(4))
print("Min:", sines.min().round(4), "Max:", sines.max().round(4))
# Sine oscillates between -1 and +1; applying it to the full circle gives one period


## Item 2 Solutions — Indexing and Slicing

In [ ]:
# Exercise 1: arange and indexing
v = np.arange(10, 20)
print("Index 3:", v[3])       # 13
print("Last element:", v[-1]) # 19


In [ ]:
# Exercise 2: slicing
v = np.arange(10, 20)
print("v[3:7]:", v[3:7])    # [13, 14, 15, 16]
print("Every other:", v[::2])  # [10, 12, 14, 16, 18]


In [ ]:
# Exercise 3: 2D indexing
data = np.array([
    [2.0, 5.0, 8.0, 6.0, 3.0],
    [0.0, 1.0, 12.0, 9.0, 0.0],
    [4.0, 4.0, 4.0, 4.0, 4.0],
])
print("Column 3:", data[:, 3])           # all rows, column index 3: [6. 9. 4.]
print("Sub-matrix:
", data[0:2, 1:4])  # rows 0-1, cols 1-3


In [ ]:
# Exercise 4: boolean mask
v = np.arange(10, 20)
mask = v > 15
print("Mask:", mask)
print("Values > 15:", v[mask])   # [16, 17, 18, 19]


In [ ]:
# Exercise 5: fancy indexing
v = np.arange(10, 20)
print("Elements at 0, 4, 9:", v[[0, 4, 9]])   # [10, 14, 19]


In [ ]:
# Exercise 6 (Stretch): any above 10
data = np.array([
    [2.0, 5.0, 8.0, 6.0, 3.0],
    [0.0, 1.0, 12.0, 9.0, 0.0],
    [4.0, 4.0, 4.0, 4.0, 4.0],
])
has_high = np.any(data > 10, axis=1)   # True for each row that has ANY value > 10
print("Rows with value > 10:", has_high)
print("Selected rows:
", data[has_high])


## Item 3 Solutions — Operations and Broadcasting

In [ ]:
# Exercise 1: element-wise arithmetic
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])
print("a + b:", a + b)   # [11, 22, 33, 44]
print("b - a:", b - a)   # [ 9, 18, 27, 36]
print("a * b:", a * b)   # [10, 40, 90, 160]
print("b / a:", b / a)   # [10., 10., 10., 10.]


In [ ]:
# Exercise 2: subtract resting potential
v = np.array([-70., -65., -55., 10., 30., -72.])
centred = v - (-70.0)
print("Centred:", centred)
# Biologically: values now show deviation from rest. 0 = at rest; positive = depolarised;
# negative = hyperpolarised. Makes the action potential shape clearer.


In [ ]:
# Exercise 3: broadcasting with (3,4) + (4,) and + (3,)
m = np.ones((3, 4))
row_vec = np.array([1., 2., 3., 4.])   # length 4: matches columns -> broadcasts
col_vec = np.array([1., 2., 3.])        # length 3: does NOT match columns (3 != 4) -> error

print("m + row_vec:
", m + row_vec)   # works: each row has 1,2,3,4 added to it

try:
    print(m + col_vec)
except ValueError as e:
    print("Error:", e)   # operands could not be broadcast together


In [ ]:
# Exercise 4: dot product by hand and NumPy
a = np.array([1., 0., -1.])
b = np.array([3., 5.,  2.])
# By hand: 1*3 + 0*5 + (-1)*2 = 3 + 0 - 2 = 1
print("By hand: 1*3 + 0*5 + (-1)*2 =", 1*3 + 0*5 + (-1)*2)
print("np.dot:", np.dot(a, b))


In [ ]:
# Exercise 5: min-max normalisation
rates = np.array([2., 5., 8., 3., 11.])
normalised = (rates - rates.min()) / (rates.max() - rates.min())
print("Normalised:", normalised)
print("Min:", normalised.min(), "Max:", normalised.max())  # 0.0 and 1.0


In [ ]:
# Exercise 6 (Stretch): outer product with broadcasting and dot product of rows
col_index = np.arange(5)           # shape (5,)
row_index  = np.arange(4).reshape(4, 1)  # shape (4, 1)
matrix = row_index * col_index     # broadcasts to (4, 5)
print("Matrix:
", matrix)
# row 1: [0,1,2,3,4]  row 3: [0,3,6,9,12]
dot_1_3 = np.dot(matrix[1], matrix[3])
print("Dot product of rows 1 and 3:", dot_1_3)  # 0+3+12+27+48 = 90


## Item 4 Solutions — Aggregation and Statistical Functions

In [ ]:
# Exercise 1: summary statistics
scores = np.array([88, 72, 95, 61, 79, 84, 91])
print("Mean:", np.mean(scores).round(2))
print("Median:", np.median(scores))
print("Min:", np.min(scores), "Max:", np.max(scores))
print("Std:", np.std(scores).round(2))
print("Index of max:", np.argmax(scores))   # index 2 (value 95)


In [ ]:
# Exercise 2: axis sums
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])
col_sums = np.sum(matrix, axis=0)   # shape (3,)
row_sums = np.sum(matrix, axis=1)   # shape (4,)
print("Column sums:", col_sums)
print("Row sums:", row_sums)
print("Total:", np.sum(matrix))
print("Sum of col_sums:", np.sum(col_sums), "== Total?", np.sum(col_sums) == np.sum(matrix))
print("Sum of row_sums:", np.sum(row_sums), "== Total?", np.sum(row_sums) == np.sum(matrix))


In [ ]:
# Exercise 3: ISIs from spike times
spike_times = np.array([5, 18, 34, 50, 51, 90])
isi = np.diff(spike_times)
print("ISIs:", isi)         # [13, 16, 16, 1, 39]
print("Min ISI:", isi.min())  # 1 ms -- very short, near the refractory period
# A 1 ms ISI suggests the neuron fired two spikes back-to-back (a burst)


In [ ]:
# Exercise 4: cumsum of ones
print(np.cumsum(np.array([1, 1, 1, 1, 1])))   # [1, 2, 3, 4, 5]
# Cumulative sum of 1s gives the counting numbers


In [ ]:
# Exercise 5: argmax per neuron
data = np.array([
    [2.0, 5.0, 8.0, 6.0, 3.0],
    [0.0, 1.0, 12.0, 9.0, 0.0],
    [4.0, 4.0, 4.0, 4.0, 4.0],
])
peak_time_per_neuron = np.argmax(data, axis=1)
print("Time index of peak per neuron:", peak_time_per_neuron)
# Neuron 0: index 2 (8.0), Neuron 1: index 2 (12.0), Neuron 2: index 0 (tie, first wins)


In [ ]:
# Exercise 6 (Stretch): z-score normalisation per neuron
data = np.array([
    [2.0, 5.0, 8.0, 6.0, 3.0],
    [0.0, 1.0, 12.0, 9.0, 0.0],
    [4.0, 4.0, 4.0, 4.0, 4.0],
])
mean_per_neuron = np.mean(data, axis=1, keepdims=True)   # shape (3,1)
std_per_neuron  = np.std(data, axis=1, keepdims=True)    # shape (3,1)
z_scored = (data - mean_per_neuron) / std_per_neuron     # broadcasts (3,5)
print("Z-scored data:
", z_scored.round(3))
print("Row means after z-scoring:", z_scored.mean(axis=1).round(10))  # ~0
print("Row stds after z-scoring:", z_scored.std(axis=1).round(10))    # ~1
# Note: neuron 2 has std=0 (all values equal 4.0), causing division by zero -> nan


## Item 5 Solutions — Random Numbers

In [ ]:
# Exercise 1: reproducibility with seed
np.random.seed(99)
first  = np.random.rand(5)
np.random.seed(99)
second = np.random.rand(5)
print("First: ", first)
print("Second:", second)
print("Identical:", np.all(first == second))   # True


In [ ]:
# Exercise 2: randn statistics
np.random.seed(0)
samples = np.random.randn(1000)
print("Mean:", samples.mean().round(3))   # should be ~0
print("Std:", samples.std().round(3))     # should be ~1
# With 1000 samples, expect to be within ~0.05 of theoretical values


In [ ]:
# Exercise 3: Poisson simulation
np.random.seed(0)
counts = np.random.poisson(lam=5, size=50)
print("First 10:", counts[:10])
print("Sample mean:", counts.mean().round(3))    # ~5
print("Sample std:", counts.std().round(3))      # ~sqrt(5) ~ 2.236
print("Theoretical std:", np.sqrt(5).round(3))   # 2.236


In [ ]:
# Exercise 4: randint max and argmax
np.random.seed(42)
ints = np.random.randint(0, 100, size=10)
print("Values:", ints)
print("Max:", ints.max(), "at index:", ints.argmax())


In [ ]:
# Exercise 5: random group assignment
np.random.seed(5)
all_indices = np.arange(20)
group_a = np.random.choice(all_indices, size=10, replace=False)
group_b = np.setdiff1d(all_indices, group_a)   # all indices not in group_a
print("Group A:", np.sort(group_a))
print("Group B:", np.sort(group_b))
print("Sizes:", len(group_a), len(group_b))


In [ ]:
# Exercise 6 (Stretch): Poisson spike train
np.random.seed(0)
n_bins      = 1000         # 1000 ms = 1 second at 1 ms resolution
target_rate = 20           # Hz
p_spike     = target_rate / n_bins   # probability per bin = 20/1000 = 0.02

spike_train = np.random.rand(n_bins) < p_spike   # boolean array: True = spike
total_spikes = np.sum(spike_train)
firing_rate  = total_spikes / 1.0    # per second (duration = 1 s)

print("Total spikes:", total_spikes)
print("Estimated firing rate:", firing_rate, "Hz")
print("Target:", target_rate, "Hz -- difference:", abs(firing_rate - target_rate), "Hz")
# With 1000 bins the estimate is noisy; expect to be within a few Hz of 20


---
# Notebook 4 Complete

You now have the core NumPy toolkit:

- **Creating arrays** -- `np.array`, `zeros`, `ones`, `full`, `arange`, `linspace`
- **Indexing and slicing** -- 1D, 2D, boolean, and fancy indexing
- **Operations and broadcasting** -- element-wise arithmetic, scalar and shape
  broadcasting, dot products
- **Aggregation** -- `sum`, `mean`, `std`, `min`, `max`, `median`, `argmin`,
  `argmax`, `cumsum`, `diff`
- **Random numbers** -- `seed`, `rand`, `randn`, `normal`, `poisson`, `randint`,
  `choice`

**Next:** Notebook 5 -- Matplotlib. You will use NumPy arrays as the data source
for every plot, so everything here applies directly.
